# Al-Zahraa Pharmacy: business questions analysis

Reads the **clean** sheet from `data/clean/`. Returns are included with their negative values (net revenue).

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 100)

CLEAN = r"D:\pharmacy_data\data\clean\alzahraa_sales_clean.csv"
c = pd.read_csv(CLEAN, dtype={"customer_phone": str}, parse_dates=["sale_date"])
c["month"] = c.sale_date.dt.to_period("M")
print(c.shape, "| total revenue:", round(c.line_total.sum(), 2))

## Q1: monthly revenue per branch, which one is better, and is the gap constant?

In [ ]:
K, M = "Kharga", "Mut - Dakhla"

q1 = c.pivot_table(index="month", columns="branch", values="line_total", aggfunc="sum").round(0)
inv = c.groupby(["month", "branch"]).invoice_id.nunique().unstack()

q1["gap"] = q1[K] - q1[M]                       # gap in EGP
q1["ratio"] = (q1[K] / q1[M]).round(2)           # how many times the Mut revenue the Kharga revenue is
q1["invoices_K"], q1["invoices_M"] = inv[K], inv[M]
q1["avg_invoice_K"] = (q1[K] / inv[K]).round(0)  # average invoice
q1["avg_invoice_M"] = (q1[M] / inv[M]).round(0)
display(q1)

print("Total:", q1[[K, M, "gap"]].sum().to_dict())
print("Ratio: min", q1.ratio.min(), "| median", q1.ratio.median(), "| max", q1.ratio.max())

q1[[K, M]].plot(marker="o", figsize=(10, 4), title="Monthly revenue by branch (EGP)");

## Q2: top 10 products per branch (revenue and units) + stagnant products

In [ ]:
last_day = c.sale_date.max()
months = c.month.nunique()

q2 = (c.groupby(["branch", "product_name", "category"])
        .agg(revenue=("line_total", "sum"), units=("quantity", "sum"),
             lines=("line_total", "size"), last_sale=("sale_date", "max"))
        .reset_index())
q2["revenue"] = q2.revenue.round(0)
q2["share_pct"] = (q2.revenue / q2.groupby("branch").revenue.transform("sum") * 100).round(1)
q2["rank_revenue"] = q2.groupby("branch").revenue.rank(ascending=False).astype(int)
q2["rank_units"] = q2.groupby("branch").units.rank(ascending=False, method="first").astype(int)
q2["units_per_month"] = (q2.units / months).round(1)
q2["days_since_last_sale"] = (last_day - q2.last_sale).dt.days

for b, x in q2.groupby("branch"):
    print("\n" + "=" * 70, "\n🏪", b, "\n" + "=" * 70)
    print("▶ Top 10 by revenue:")
    display(x.nsmallest(10, "rank_revenue")[["product_name", "revenue", "share_pct", "units", "rank_units"]])
    print("▶ Top 10 by units:")
    display(x.nsmallest(10, "rank_units")[["product_name", "units", "revenue", "rank_revenue"]])
    print("▶ Lowest 8 by movement:")
    display(x.nsmallest(8, "units")[["product_name", "units", "units_per_month", "revenue", "days_since_last_sale"]])

## Q3: seasons: cold medicines, and ORS / Antinal / sunscreen

In [ ]:
s = c[~c.is_return]
groups = {
    "Cold & Flu": s.category == "Cold & Flu",
    "ORS": s.product_name.str.contains("ORS"),
    "Antinal": s.product_name.str.contains("Antinal"),
    "Sunscreen": s.product_name.str.contains("Nivea Sun"),
}
q3 = pd.DataFrame({k: s[m].groupby("month").quantity.sum() for k, m in groups.items()}).fillna(0).astype(int)
display(q3)

# seasonality index: month units / average month (1 = a normal month)
display((q3 / q3.mean()).round(2))

q3.plot(marker="o", figsize=(10, 4), title="Units sold per month");

## Q4: the four pharmacists (a fair comparison that accounts for the shift)

In [ ]:
s = c[~c.is_return].copy()
s["gross"] = s.quantity * s.unit_price          # before discount
s["disc_cost"] = s.gross - s.line_total          # discount cost in EGP

q4 = s.groupby(["branch", "pharmacist", "shift"]).agg(
    revenue=("line_total", "sum"), invoices=("invoice_id", "nunique"), lines=("line_total", "size"),
    gross=("gross", "sum"), disc_cost=("disc_cost", "sum"),
    pct_lines_discounted=("discount_pct", lambda x: (x > 0).mean() * 100))
q4["avg_invoice"] = q4.revenue / q4.invoices
q4["disc_rate_pct"] = q4.disc_cost / q4.gross * 100            # discount as a share of sales
q4["disc_cost_per_year"] = q4.disc_cost * 12 / c.month.nunique()
display(q4.round(1))

print("▶ Discount distribution per pharmacist (% of their lines):")
display((pd.crosstab(s.pharmacist, s.discount_pct, normalize="index") * 100).round(1))

## Q5: busy hours and days + Ramadan and holidays

Holiday dates are approximate (calendar based); Ramadan 2026 is 18 Feb - 19 Mar as given by the client.

In [ ]:
s = c[~c.is_return]
inv5 = (s.groupby("invoice_id")
          .agg(date=("sale_date", "first"), time=("sale_time", "first"), branch=("branch", "first"),
               revenue=("line_total", "sum")).reset_index())
inv5["hour"] = inv5.time.str[:2].astype(int)
inv5["ramadan"] = inv5.date.between("2026-02-18", "2026-03-19")

def per_day_by_hour(x):
    """Average invoices per hour per working day"""
    return (x.groupby(["hour", "branch"]).size().unstack().fillna(0) / x.groupby("branch").date.nunique()).round(2)

print("▶ Invoices per hour (normal day):");  display(per_day_by_hour(inv5[~inv5.ramadan]).T)
print("▶ Invoices per hour (Ramadan 2026):"); display(per_day_by_hour(inv5[inv5.ramadan]).T)

days5 = inv5.groupby(["branch", "date"]).agg(invoices=("invoice_id", "size"), revenue=("revenue", "sum")).reset_index()
days5["weekday"] = days5.date.dt.day_name()
print("▶ Average invoices per day by weekday:")
display(days5.groupby(["branch", "weekday"]).invoices.mean().round(1).unstack())

days5["ramadan"] = days5.date.between("2026-02-18", "2026-03-19")
print("▶ Ramadan vs normal (daily average):")
display(days5.groupby(["branch", "ramadan"])[["invoices", "revenue"]].mean().round(1))

holidays = {"Eid al-Adha 2025": ("2025-06-06", "2025-06-09"), "Eid al-Fitr 2026": ("2026-03-20", "2026-03-22"),
            "Eid al-Adha 2026": ("2026-05-27", "2026-05-30")}
print("▶ Holidays (average invoices per day):")
display(pd.DataFrame({k: days5[days5.date.between(a, b)].groupby("branch").invoices.mean().round(1)
                      for k, (a, b) in holidays.items()}))

## Q6: effect of the January 2026 price increase on quantities and revenue

In [ ]:
s = c[~c.is_return]

# 1) is the increase really 12%? (base price per product before and after)
base = s.groupby(["product_name", s.sale_date >= "2026-01-01"]).unit_price.min().unstack()
print("Price ratio after/before per product:", (base[True] / base[False]).round(3).value_counts().to_dict())

# 2) same-months comparison: Jun-Sep 2025 vs Jun-Sep 2026
def kpis(x):
    return pd.Series({"revenue": x.line_total.sum(), "units": x.quantity.sum(),
                      "invoices": x.invoice_id.nunique(),
                      "revenue_per_unit": x.line_total.sum() / x.quantity.sum()})

summer = s[s.sale_date.dt.month.between(6, 9)]
before, after = summer[summer.sale_date.dt.year == 2025], summer[summer.sale_date.dt.year == 2026]

def compare(a, b):
    t = pd.DataFrame({"Jun-Sep 2025": kpis(a), "Jun-Sep 2026": kpis(b)})
    t["change_%"] = ((t.iloc[:, 1] / t.iloc[:, 0] - 1) * 100).round(1)
    return t.round(1)

print("▶ Both branches:"); display(compare(before, after))
for b in s.branch.unique():
    print("▶", b); display(compare(before[before.branch == b], after[after.branch == b]))

## Q7: share of InstaPay and wallets month by month

In [ ]:
s = c[~c.is_return]
inv7 = (s.groupby("invoice_id")
          .agg(month=("month", "first"), pay=("payment_method", "first"), branch=("branch", "first"),
               revenue=("line_total", "sum")).reset_index())
print("Unknown invoices (excluded from the denominator):", (inv7.pay == "Unknown").sum())
known = inv7[inv7.pay != "Unknown"]

q7 = (pd.crosstab(known.month, known.pay, normalize="index") * 100).round(1)   # % of invoice count
q7["electronic"] = (q7.InstaPay + q7.Wallet).round(1)
q7["invoices"] = known.groupby("month").size()
display(q7)

print("▶ By branch (%):")
display((pd.crosstab(known.branch, known.pay, normalize="index") * 100).round(1))

q7[["InstaPay", "Wallet", "electronic"]].plot(marker="o", figsize=(10, 4), title="Electronic payment share of invoices (%)");

## Q9: share of antibiotics per branch, and does it rise in winter?

(Q8 is answered by the expiry report: `scripts/expiry_report.py`)

In [ ]:
s = c[~c.is_return].copy()
s["is_ab"] = s.category == "Antibiotics"
s["season"] = np.where(s.sale_date.dt.month.isin([10, 11, 12, 1, 2, 3]), "winter (Oct-Mar)", "rest of year")

def ab_share(g):
    return pd.Series({"lines_%": g.is_ab.mean() * 100,
                      "units_%": g.quantity[g.is_ab].sum() / g.quantity.sum() * 100,
                      "revenue_%": g.line_total[g.is_ab].sum() / g.line_total.sum() * 100,
                      "ab_units": g.quantity[g.is_ab].sum()})

print("▶ Per branch (whole period):");  display(s.groupby("branch").apply(ab_share).round(1))
print("▶ Per branch x season:");         display(s.groupby(["branch", "season"]).apply(ab_share).round(1))

q9 = s.groupby(["month", "branch"]).is_ab.mean().mul(100).unstack().round(1)
q9["both"] = s.groupby("month").is_ab.mean().mul(100).round(1)
display(q9)
q9.plot(marker="o", figsize=(10, 4), title="Antibiotics share of sale lines (%)");

## Q10: revenue forecast per branch, Oct-Dec 2026 (with a range)

**Method:** revenue of the same month last year x the annual growth rate (computed from Jun-Sep, the months present in both years).
**Range:** growth rate +/- its month-to-month volatility.

In [ ]:
rev = c.pivot_table(index="month", columns="branch", values="line_total", aggfunc="sum")
P = pd.Period
overlap = [6, 7, 8, 9]            # months present in both years
targets = [10, 11, 12]            # months to forecast

rows = []
for b in rev.columns:
    x = rev[b]
    # annual growth: same months 2026 / 2025
    monthly_growth = pd.Series({m: x[P(f"2026-{m:02d}")] / x[P(f"2025-{m:02d}")] for m in overlap})
    g = sum(x[P(f"2026-{m:02d}")] for m in overlap) / sum(x[P(f"2025-{m:02d}")] for m in overlap)
    spread = monthly_growth.std()          # month-to-month volatility of growth = source of the range
    print(f"{b}: growth {g:.3f} | monthly {monthly_growth.round(2).to_dict()} | spread ±{spread:.3f}")
    for m in targets:
        last_year = x[P(f"2025-{m:02d}")]
        rows.append({"branch": b, "month": f"2026-{m:02d}", "last_year": last_year,
                     "low": last_year * (g - spread), "forecast": last_year * g, "high": last_year * (g + spread)})

q10 = pd.DataFrame(rows).round(0)
display(q10)
print("▶ Quarter total (Oct-Dec 2026):")
display(q10.groupby("branch")[["last_year", "low", "forecast", "high"]].sum().round(0))